In [9]:
from tqdm import tqdm
from architecture import Embeddings, PositionalEncoding, Encoder, Batch,BinaryClassifier, PGKPR
import torch.nn as nn
import torch
import math
from utils import loadVocab, vocab_path

from torch.utils.data import TensorDataset, DataLoader, RandomSampler

In [10]:
train_inputs,val_inputs, test_inputs = torch.load("train_inputs.pt"),torch.load("val_inputs.pt"),torch.load("test_inputs.pt")
train_labels,val_labels, test_labels = torch.load("train_labels.pt"),torch.load("val_labels.pt"),torch.load("test_labels.pt")


In [11]:
LOWERCASE = True
token2idx, idx2token = loadVocab(vocab_path, lowercase=LOWERCASE) # I should go back to the first stage and understand better how the vocab is built. However, it is not subtokened
vocab_size = len(token2idx)


In [12]:
# # Encode data

# #Initialize
# encoder = Encoder(num_layers=6, d_model=512, nhead=8, d_ff=2048, dropout=0.1, return_layer=6)

# #Manual run 
# embed_test = Embeddings(512, vocab_size)
# pos_encod_test = PositionalEncoding(512, 0.1, double=False)

# src_embed_pos = nn.Sequential(Embeddings(512, vocab_size), #Token id to embedding
#                                            PositionalEncoding(512, 0.1, double=False))

# ecbplus_mini_inputs_mask = (ecbplus_mini_inputs != 0).unsqueeze(-2)

# xx = src_embed_pos.forward(ecbplus_mini_inputs)
# x = embed_test.forward(ecbplus_mini_inputs)
# y  = pos_encod_test.forward(x)

# final_encoder_layer, intermediate_encoder_layer = encoder.forward(y,ecbplus_mini_inputs_mask)

# ## 'Decode' Part for CDCR.

# binary_classifier = BinaryClassifier(hidden_size=512, layer_norm_eps=1e-12)
# output = binary_classifier(final_encoder_layer)



In [13]:
# model
#D_model is hidden_state size
D_MODEL = 64 #Change temporariliy to a small size from 512 to run in small laptop

MID_LOSS = True
DOUBLE_POSITION = False
tqdm_print_step = 1
model_save_path = 'models/'
BATCH_SIZE = 8
VAL_BATCH_SIZE = 8 

In [14]:
# Load model and plug optimizer
model = PGKPR(d_model=D_MODEL, vocab=vocab_size, mid_loss=MID_LOSS, double_pe=DOUBLE_POSITION,
                          shared_emb=True)


model_opt = torch.optim.Adam(model.parameters(), lr=1e-4, betas=(0.9, 0.98), eps=1e-9)

/home/sago/Documents/GitHub/CDCR_lexical_diversity/MTL/PGKPR_adapted/architecture.py:323: UserWarning: nn.init.xavier_uniform is now deprecated in favor of nn.init.xavier_uniform_.
  nn.init.xavier_uniform(p)


In [15]:
# I will continue to analyze whether I need that Batch masking which seems to be more for next sentence prediction
# It is indeed like that but I like the concept of the batch doing the marking. I will replicate it 

In [16]:
import random
#Pack to tuples 
train_data = train_inputs, train_labels 
val_data = val_inputs, val_labels
test_data = test_inputs, test_labels

#Subset to run locally
subset_size = 64
random.seed(20)
random_indices_train = random.sample(range(len(train_inputs)), subset_size)

random.seed(30)
random_indices_val = random.sample(range(len(val_inputs)), subset_size)

# Create a new subset TensorDataset using the random indices
subset_train_dataset = TensorDataset(train_inputs[random_indices_train], train_labels[random_indices_train])
subset_val_dataset = TensorDataset(val_inputs[random_indices_val], val_labels[random_indices_val])

# train_data = TensorDataset(*train_data)
train_sampler = RandomSampler(subset_train_dataset)
train_dataloader = DataLoader(subset_train_dataset, sampler=train_sampler, batch_size=BATCH_SIZE)

# val_data = TensorDataset(*val_data)
# test_data = TensorDataset(*test_data)

val_dataloader = DataLoader(subset_val_dataset, batch_size=VAL_BATCH_SIZE)
test_dataloader = DataLoader(subset_val_dataset, batch_size=VAL_BATCH_SIZE)  

dataloader = {'train': train_dataloader, 'val': val_dataloader, 'test':test_dataloader}

In [17]:
def run_once(model, batch,loss_function,optimizer,is_training):
    predicted_probs= model.forward(src=batch.src,src_mask=batch.src_mask)
    predicted_probs = predicted_probs.squeeze()

    loss = loss_function(predicted_probs, batch.trg)
    
    if is_training and optimizer is not None:
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    return loss

loss_function = torch.nn.BCELoss()
best_loss = float("inf")

for epoch in range(5): #TODO define epochs parameter from config file
    # scheduler.step()
    print(f'\n EPOCH {epoch}')
    model.train()
    total_loss = 0
    #cdcr_loss, pi_loss= 0, 0
    
    for phase in ['train', 'val']:
        if phase=='train':
            model.train()
            is_training= True
        else:
            model.eval()
            is_training= False

        with tqdm(total=len(dataloader[phase])) as pbar: 
            #Dataloader should be controlled to send a batch of one task only.
            for i, data in enumerate(dataloader[phase]):
                src, tgt= data #dataloader must have source and target, in my case the label
                batch = Batch(src, tgt)
                # src, tgt= src.cuda() tgt.cuda()#TODO send everything to device. I could adapt it to finding device first
                binary_loss = run_once(model,batch,loss_function,model_opt,is_training)
                total_loss += binary_loss #It would be a good idea to monitor both losses separately
                
                if i % tqdm_print_step == 0:
                        pbar.update(tqdm_print_step)
                        pbar.set_description(
                            f'train loss: {binary_loss:.4f}')
                        
        epoch_loss = total_loss / len(dataloader[phase]) #Mean relative to dataloader size

        print(f"epoch loss: {epoch_loss}")

        if phase == 'val' and (epoch_loss < best_loss or epoch % 10 == 0) and epoch > 5:
            print("save model")
            best_loss = epoch_loss
            torch.save(model.state_dict(), f'{model_save_path}/epoch{epoch}_loss_{epoch_loss}.pkl')



 EPOCH 0


  0%|          | 0/8 [00:00<?, ?it/s]

train loss: 0.3180: 100%|██████████| 8/8 [00:07<00:00,  1.14it/s]


epoch loss: 0.5269144773483276


train loss: 0.7908: 100%|██████████| 8/8 [00:03<00:00,  2.43it/s]


epoch loss: 1.315199613571167

 EPOCH 1


train loss: 0.3641:  12%|█▎        | 1/8 [00:01<00:13,  1.96s/it]

: 

: 

In [ ]:
# Types seem to be fine, now I have to add the the optimizer step.